In [0]:
from pathlib import Path
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as f
from torch.utils.data import IterableDataset, DataLoader
import pyarrow.parquet as pq
from collections import defaultdict
import pickle
import json

In [0]:
# ============================================================
# Parquet Streaming Dataset
# ============================================================
class ParquetTwoTowerDataset(IterableDataset):
    def __init__(self, parquet_dir,
                 user_id_col, post_id_col,
                 user_cat_cols, user_num_cols,
                 post_cat_cols, post_num_cols,
                 user_emb_col, post_emb_col,
                 label_col,
                 batch_size=1024):
        """
        parquet_dir: path to the joined parquet (after Spark preprocessing)
        """
        super().__init__()
        self.user_id_col, self.post_id_col = user_id_col, post_id_col
        self.user_cat_cols, self.user_num_cols = user_cat_cols, user_num_cols
        self.post_cat_cols, self.post_num_cols = post_cat_cols, post_num_cols
        self.user_emb_col, self.post_emb_col = user_emb_col, post_emb_col
        self.label_col = label_col
        self.batch_size = batch_size

        self.parquet_files = list(Path(parquet_dir).glob("*.parquet"))

    def __iter__(self):
        for parquet_file in self.parquet_files:
            pf = pq.ParquetFile(parquet_file)
            for batch in pf.iter_batches(batch_size=self.batch_size):
                pdf = batch.to_pandas()
                # Convert to torch tensors
                user_id = torch.tensor(pdf[self.user_id_col].values, dtype=torch.long)
                post_id = torch.tensor(pdf[self.post_id_col].values, dtype=torch.long)
                labels = torch.tensor(pdf[self.label_col].values, dtype=torch.float32)

                user_cat = torch.stack([torch.tensor(pdf[c].values, dtype=torch.long) for c in self.user_cat_cols], dim=1)
                user_num = torch.stack([torch.tensor(pdf[c].values, dtype=torch.float32) for c in self.user_num_cols], dim=1)

                post_cat = torch.stack([torch.tensor(pdf[c].values, dtype=torch.long) for c in self.post_cat_cols], dim=1)
                post_num = torch.stack([torch.tensor(pdf[c].values, dtype=torch.float32) for c in self.post_num_cols], dim=1)

                # embeddings stored as arrays in parquet → each row is numpy array
                lastn_embs = torch.tensor(np.stack(pdf[self.user_emb_col].values), dtype=torch.float32)
                post_emb = torch.tensor(np.stack(pdf[self.post_emb_col].values), dtype=torch.float32)

                yield {
                    "user_id": user_id,
                    "user_cat": user_cat,
                    "user_num": user_num,
                    "lastn_embs": lastn_embs,
                    "post_id": post_id,
                    "post_cat": post_cat,
                    "post_num": post_num,
                    "post_emb": post_emb,
                    "label": labels,
                }

In [0]:
# ============================================================
# Model
# ============================================================
class MLP(nn.Module):
    """
    Multi-Layer Perceptron (MLP) used to process concatenated feature vectors.
    Consists of several linear layers with ReLU activations and dropout for regularization.
    """
    def __init__(self, in_dim, hidden_dims=(256,128), dropout=0.2):
        super().__init__()
        layers = []  # List to hold layers
        prev = in_dim  # Track input dimension for each layer
        for h in hidden_dims:
            # Add a Linear layer, followed by ReLU and Dropout
            layers += [nn.Linear(prev, h), nn.ReLU(), nn.Dropout(dropout)]
            prev = h  # Update input dimension for next layer
        self.net = nn.Sequential(*layers)  # Combine layers into a sequential model

    def forward(self, x):
        # Forward pass through the MLP
        return self.net(x)

class UserTower(nn.Module):
    """
    Encodes user features into a dense vector representation.
    Combines user ID, categorical features, numerical features, and sequence embeddings (e.g., user history).
    """
    def __init__(self, uid_vocab, uid_dim, user_cat_dims, user_num_dim, lastn_emb_dim, hidden_dims, dropout):
        super().__init__()
        # Embedding for user ID (with padding index 0)
        self.user_id_emb = nn.Embedding(uid_vocab+1, uid_dim, padding_idx=0)
        # Embeddings for each categorical user feature
        self.user_cat_embs = nn.ModuleList([nn.Embedding(v+1, d, padding_idx=0) for v, d in user_cat_dims])
        # MLP to combine all user features into a single vector
        self.mlp = MLP(uid_dim + sum(d for _, d in user_cat_dims) + user_num_dim + lastn_emb_dim, hidden_dims, dropout)

    def forward(self, user_id, user_cat, user_num, lastn_embs):
        # Embed user ID
        u_id = self.user_id_emb(user_id)
        # Embed and concatenate all categorical features (if any)
        if self.user_cat_embs:
            u_cat = torch.cat([emb(user_cat[:, i]) for i, emb in enumerate(self.user_cat_embs)], dim=-1)
        else:
            # If no categorical features, create empty tensor
            u_cat = torch.zeros(u_id.size(0), 0, device=u_id.device)
        # Concatenate all features and pass through MLP
        return self.mlp(torch.cat([u_id, u_cat, user_num, lastn_embs], dim=-1))

class ItemTower(nn.Module):
    """
    Encodes post/item features into a dense vector representation.
    Combines post ID, categorical features, numerical features, and content embeddings.
    """
    def __init__(self, pid_vocab, pid_dim, post_cat_dims, post_num_dim, post_emb_dim, hidden_dims, dropout):
        super().__init__()
        # Embedding for post ID (with padding index 0)
        self.post_id_emb = nn.Embedding(pid_vocab+1, pid_dim, padding_idx=0)
        # Embeddings for each categorical post feature
        self.post_cat_embs = nn.ModuleList([nn.Embedding(v+1, d, padding_idx=0) for v, d in post_cat_dims])
        # MLP to combine all post features into a single vector
        self.mlp = MLP(pid_dim + sum(d for _, d in post_cat_dims) + post_num_dim + post_emb_dim, hidden_dims, dropout)

    def forward(self, post_id, post_cat, post_num, post_emb):
        # Embed post ID
        p_id = self.post_id_emb(post_id)
        # Embed and concatenate all categorical features (if any)
        if self.post_cat_embs:
            p_cat = torch.cat([emb(post_cat[:, i]) for i, emb in enumerate(self.post_cat_embs)], dim=-1)
        else:
            # If no categorical features, create empty tensor
            p_cat = torch.zeros(p_id.size(0), 0, device=p_id.device)
        # Concatenate all features and pass through MLP
        return self.mlp(torch.cat([p_id, p_cat, post_num, post_emb], dim=-1))

def _last_linear_out_features(seq):
    """
    Utility to get output dimension of last Linear layer in a Sequential.
    """
    for l in reversed(seq):
        if isinstance(l, nn.Linear): return l.out_features
    raise ValueError("No Linear found")

class TwoTowerModel(nn.Module):
    """
    Combines user and item towers, projects them to a common space,
    and computes similarity scores for retrieval.
    """
    def __init__(self, user_tower, item_tower, proj_dim, normalize=True, temperature=0.1):
        super().__init__()
        self.user_tower, self.item_tower = user_tower, item_tower
        self.user_proj = nn.Linear(_last_linear_out_features(user_tower.mlp.net), proj_dim)
        self.item_proj = nn.Linear(_last_linear_out_features(item_tower.mlp.net), proj_dim)
        self.normalize = normalize
        self.temperature = temperature
    def encode_user(self,*a):
        z = self.user_proj(self.user_tower(*a)); return f.normalize(z,dim=-1) if self.normalize else z
    def encode_item(self,*a):
        z = self.item_proj(self.item_tower(*a)); return f.normalize(z,dim=-1) if self.normalize else z
    def forward(self, user_inputs, item_inputs):
        # Computes similarity scores between user and item vectors
        u = self.encode_user(*user_inputs); v = self.encode_item(*item_inputs)
        return (u @ v.t())/self.temperature

In [0]:
# ============================================================
# Losses & Trainer
# ============================================================

# ============================================================
# Enhanced Losses & Trainer with Negative Sampling
# ============================================================
def pointwise_bce_loss(scores, labels):
    """
    Binary cross-entropy loss for pointwise objectives.
    """
    return f.binary_cross_entropy_with_logits(scores, labels.float())

def bpr_loss(u_vecs, pos_vecs, neg_vecs):
    """
    Bayesian Personalized Ranking (BPR) loss for pairwise ranking.
    """
    pos_scores = (u_vecs*pos_vecs).sum(dim=-1)
    if neg_vecs.dim()==3:
        neg_scores = (u_vecs.unsqueeze(1)*neg_vecs).sum(dim=-1)
        return -f.logsigmoid(pos_scores.unsqueeze(1)-neg_scores).mean()
    neg_scores = (u_vecs*neg_vecs).sum(dim=-1)
    return -f.logsigmoid(pos_scores-neg_scores).mean()

def bpr_inbatch_loss(u_vecs, pos_vecs, post_log_probs=None):
    """
    Memory-efficient BPR with in-batch negatives using similarity matrix.
    For each sample i: -logsigmoid(s_i,i - s_i,j) over j != i
    With optional bias correction: cos(u, p_i) - log(p_i)
    """
    if post_log_probs is not None:
        # Apply bias correction: cos(u, p_i) - log(p_i)
        logits = u_vecs @ pos_vecs.t() - post_log_probs.unsqueeze(0)  # (B, B)
    else:
        logits = u_vecs @ pos_vecs.t()  # (B, B)
    
    B = logits.size(0)
    diffs = logits.diag().unsqueeze(1) - logits  # (B, B)
    mask = ~torch.eye(B, dtype=torch.bool, device=logits.device)
    return -f.logsigmoid(diffs[mask]).mean()

def inbatch_softmax_loss(logits, post_log_probs=None):
    """
    Listwise softmax loss using in-batch negatives.
    With optional bias correction: cos(u, p_i) - log(p_i)
    """
    if post_log_probs is not None:
        # Apply bias correction
        logits = logits - post_log_probs.unsqueeze(0)  # (B, B)
    
    return f.cross_entropy(logits, torch.arange(logits.size(0), device=logits.device))

class PopularityBasedNegativeSampler:
    """
    Negative sampler that samples posts inversely proportional to their popularity.
    Sampling probability ∝ (views)^(0.75)
    """
    def __init__(self, post_df, post_id_col, views_col, power=0.75):
        """
        Initialize the negative sampler.
        
        Args:
            post_df: Spark DataFrame containing post information
            post_id_col: Column name for post IDs
            views_col: Column name for view counts
            power: Power for popularity weighting (default: 0.75)
        """
        self.power = power
        
        # Convert to pandas for easier manipulation
        post_pdf = post_df.select(post_id_col, views_col).toPandas()
        
        # Create mapping from post_id to views
        self.post_views = dict(zip(post_pdf[post_id_col], post_pdf[views_col]))
        
        # Calculate sampling weights based on popularity
        views = np.array(list(self.post_views.values()))
        # Add small epsilon to avoid division by zero
        views = np.maximum(views, 1)
        weights = views ** (self.power)
        
        # Normalize weights
        self.sampling_weights = weights / weights.sum()
        self.post_ids = list(self.post_views.keys())
        
        # Create probability distribution for sampling
        self.prob_dist = torch.tensor(self.sampling_weights, dtype=torch.float32)
    
    def sample_negatives(self, user_interactions, n_negatives_per_user, device="cpu"):
        """
        Sample negative posts for each user.
        
        Args:
            user_interactions: Dict mapping user_id to set of positive post_ids
            n_negatives_per_user: Number of negative samples per user
            device: Device to place tensors on
            
        Returns:
            Dict mapping user_id to list of negative post_ids
        """
        negative_samples = {}
        
        for user_id, positive_posts in user_interactions.items():
            # Get all posts that this user hasn't interacted with
            available_negatives = [pid for pid in self.post_ids if pid not in positive_posts]
            
            if len(available_negatives) == 0:
                # If no negatives available, sample from all posts
                available_negatives = self.post_ids
            
            # Create sampling weights for available negatives
            available_indices = [self.post_ids.index(pid) for pid in available_negatives]
            available_weights = self.prob_dist[available_indices]
            available_weights = available_weights / available_weights.sum()
            
            # Sample negatives
            if len(available_negatives) <= n_negatives_per_user:
                # If not enough negatives, sample with replacement
                sampled_indices = torch.multinomial(available_weights, n_negatives_per_user, replacement=True)
            else:
                # Sample without replacement
                sampled_indices = torch.multinomial(available_weights, n_negatives_per_user, replacement=False)
            
            negative_samples[user_id] = [available_negatives[idx] for idx in sampled_indices]
        
        return negative_samples

class TwoTowerTrainer:
    """
    Enhanced trainer that supports multiple negative sampling strategies.
    Supports different objectives and negative sampling strategies.
    """
    def __init__(self, model, dataset, objective="listwise", neg_sampling="in-batch", 
                 neg_ratio=3, lr=1e-3, device="cpu", post_df=None, post_id_col=None, 
                 views_col="views", popularity_power=0.75):
        self.model = model.to(device)
        self.dataset = dataset
        self.objective = objective
        self.neg_sampling = neg_sampling
        self.neg_ratio = neg_ratio
        self.opt = torch.optim.AdamW(self.model.parameters(), lr=lr)
        self.device = device
        
        # Initialize negative sampler if needed
        self.negative_sampler = None
        if neg_sampling == "simple" and post_df is not None:
            self.negative_sampler = PopularityBasedNegativeSampler(
                post_df, post_id_col, views_col, popularity_power
            )
        
        # Build user interactions for negative sampling
        self.user_interactions = None
        if neg_sampling == "simple" and dataset is not None:
            self.user_interactions = build_interactions(dataset)
        
        # Create post popularity log probabilities for bias correction
        self.post_log_probs = None
        if post_df is not None:
            self._compute_post_log_probs(post_df, post_id_col, views_col, popularity_power)

    def _compute_post_log_probs(self, post_df, post_id_col, views_col, power=1):
        """
        Compute log probabilities for posts based on popularity for bias correction.
        """
        post_pdf = post_df.select(post_id_col, views_col).toPandas()
        
        # Calculate popularity-based probabilities
        views = np.maximum(post_pdf[views_col].values, 1)  # Avoid division by zero
        probs = views ** (power)
        probs = probs / probs.sum()  # Normalize
        
        # Create mapping from post_id to log probability
        self.post_log_probs = dict(zip(post_pdf[post_id_col], np.log(probs)))

    def _batch_to_device(self, batch):
        # Moves batch tensors to the target device
        td = lambda x: x.to(self.device)
        return ((td(batch['user_id']), td(batch['user_cat']), td(batch['user_num']), td(batch['lastn_embs'])),
                (td(batch['post_id']), td(batch['post_cat']), td(batch['post_num']), td(batch['post_emb'])),
                td(batch['label']))

    def _get_post_log_probs_batch(self, post_ids):
        """
        Get log probabilities for a batch of post IDs.
        """
        if self.post_log_probs is None:
            return None
        
        # Convert post_ids to list if it's a tensor
        if isinstance(post_ids, torch.Tensor):
            post_ids = post_ids.cpu().tolist()
        
        # Get log probabilities for each post in the batch
        batch_log_probs = []
        for pid in post_ids:
            batch_log_probs.append(self.post_log_probs.get(pid, 0.0))
        
        return torch.tensor(batch_log_probs, device=self.device)

    def train_one_epoch(self, loader):
        """
        Trains the model for one epoch over the data loader.
        """
        self.model.train()
        total_loss = 0.0
        n_batches = 0
        
        for batch in loader:
            user_inputs, item_inputs, labels = self._batch_to_device(batch)
            self.opt.zero_grad()
            
            if self.objective == "listwise":
                if self.neg_sampling == "in-batch":
                    # Get post log probabilities for bias correction
                    post_log_probs = self._get_post_log_probs_batch(item_inputs[0])
                    logits = self.model(user_inputs, item_inputs)
                    loss = inbatch_softmax_loss(logits, post_log_probs)
                else:
                    # For simple negative sampling, we'd need to implement
                    # a more complex approach to get negative features
                    logits = self.model(user_inputs, item_inputs)
                    loss = inbatch_softmax_loss(logits)
                    
            elif self.objective == "pairwise":
                u_vecs = self.model.encode_user(*user_inputs)
                pos_vecs = self.model.encode_item(*item_inputs)
                
                if self.neg_sampling == "in-batch":
                    # Get post log probabilities for bias correction
                    post_log_probs = self._get_post_log_probs_batch(item_inputs[0])
                    loss = bpr_inbatch_loss(u_vecs, pos_vecs, post_log_probs)
                else:
                    # Simple negative sampling - for now, use in-batch as fallback
                    # In practice, you'd implement proper negative sampling here
                    loss = bpr_inbatch_loss(u_vecs, pos_vecs)
                    
            elif self.objective == "pointwise":
                if self.neg_sampling == "in-batch":
                    # Get post log probabilities for bias correction
                    post_log_probs = self._get_post_log_probs_batch(item_inputs[0])
                    
                    # Compute scores with bias correction
                    u_vecs = self.model.encode_user(*user_inputs)
                    pos_vecs = self.model.encode_item(*item_inputs)
                    scores = (u_vecs * pos_vecs).sum(dim=-1)
                    
                    if post_log_probs is not None:
                        scores = scores - post_log_probs
                    
                    # Create labels (1 for positive interactions)
                    pos_labels = torch.ones_like(scores)
                    loss = pointwise_bce_loss(scores, pos_labels)
                else:
                    # Simple negative sampling - for now, use in-batch as fallback
                    u_vecs = self.model.encode_user(*user_inputs)
                    pos_vecs = self.model.encode_item(*item_inputs)
                    pos_scores = (u_vecs * pos_vecs).sum(dim=-1)
                    pos_labels = torch.ones_like(pos_scores)
                    loss = pointwise_bce_loss(pos_scores, pos_labels)
            else:
                raise ValueError(f"Unknown objective {self.objective}")
            
            loss.backward()
            self.opt.step()
            total_loss += loss.item()
            n_batches += 1
            
        return total_loss / n_batches

In [0]:
# ============================================================
# Build interactions from dataset
# ============================================================
def build_interactions(dataset, user_id_col="user_id", post_id_col="post_id", label_col="label"):
    """
    Converts dataset interactions into a dict: user -> set(pos_items).
    Works for ParquetTwoTowerDataset.
    """
    interactions = defaultdict(set)

    if hasattr(dataset, "data_iter"):
        # Streaming dataset with data_iter attribute
        for batch in dataset.data_iter:
            users = batch[user_id_col].tolist()
            posts = batch[post_id_col].tolist()
            labels = batch[label_col].tolist()
            for u, p, l in zip(users, posts, labels):
                if l > 0:
                    interactions[u].add(p)
    elif hasattr(dataset, "__iter__") and hasattr(dataset, "parquet_files"):
        # ParquetTwoTowerDataset (IterableDataset)
        for batch in dataset:
            users = batch[user_id_col].tolist()
            posts = batch[post_id_col].tolist()
            labels = batch[label_col].tolist()
            for u, p, l in zip(users, posts, labels):
                if l > 0:
                    interactions[u].add(p)
    else:
        raise ValueError("Dataset type not supported for build_interactions")

    return interactions

In [0]:
# ============================================================
# Orchestrator Helpers
# ============================================================
def build_model_from_dims(n_users, n_posts, user_cat_dims, post_cat_dims,
                          user_num_dim, post_num_dim,
                          id_emb_dim=16, cat_emb_dim=8, emb_dim=512, proj_dim=64,
                          hidden_dims=(256, 128), dropout=0.2, temperature=0.1):
    """Construct UserTower, ItemTower and wrap in TwoTowerModel."""
    user_tower = UserTower(
        uid_vocab=int(n_users), uid_dim=id_emb_dim,
        user_cat_dims=[(c, min(cat_emb_dim, c)) for c in user_cat_dims],
        user_num_dim=user_num_dim,
        lastn_emb_dim=emb_dim,
        hidden_dims=hidden_dims,
        dropout=dropout,
    )
    item_tower = ItemTower(
        pid_vocab=int(n_posts), pid_dim=id_emb_dim,
        post_cat_dims=[(c, min(cat_emb_dim, c)) for c in post_cat_dims],
        post_num_dim=post_num_dim,
        post_emb_dim=emb_dim,
        hidden_dims=hidden_dims,
        dropout=dropout,
    )
    return TwoTowerModel(user_tower, item_tower, proj_dim=proj_dim, temperature=temperature)

def train_streaming_data_loop(trainer, train_loader, epochs=5):
    """
    Training loop specifically for streaming datasets.
    """
    print(f"Starting training for {epochs} epochs...")
    
    for epoch in range(1, epochs+1):
        # Training
        loss = trainer.train_one_epoch(train_loader)
        
        print(f"Epoch {epoch}:")
        print(f"  Training Loss: {loss:.4f}")

    return trainer.model  

def run_parquet_streaming(parquet_dir, model, trainer, 
                            user_id_col, post_id_col,
                            user_cat_cols, user_num_cols,
                            post_cat_cols, post_num_cols,
                            user_emb_col, post_emb_col,
                            label_col,
                            batch_size=1024, epochs=5):
    """
    Run in parquet streaming mode.
    """
    print("Creating datasets ...")
    
    # Create datasets
    train_dataset = ParquetTwoTowerDataset(
        parquet_dir=parquet_dir,
        user_id_col=user_id_col+'_idx', 
        post_id_col=post_id_col+'_idx',
        user_cat_cols=[c+'_idx' for c in user_cat_cols],
        user_num_cols=[c+'_norm' for c in user_num_cols],
        post_cat_cols=[c+'_idx' for c in post_cat_cols],
        post_num_cols=[c+'_norm' for c in post_num_cols],
        user_emb_col=user_emb_col,
        post_emb_col=post_emb_col,
        label_col=label_col,
        batch_size=batch_size
    )
    
    # Create data loaders
    train_loader = DataLoader(train_dataset, batch_size=None, num_workers=0)

    print(f"Train dataset: {len(train_dataset.parquet_files)} parquet files")

    # Run training with streaming evaluation
    model = train_streaming_data_loop(
        trainer,
        train_loader,
        epochs=epochs
    )

    return model

In [0]:
# ============================================================
# Orchestration
# ============================================================
def run_model_training(metadata_dir, parquet_dir, model_args, trainer_args, data_args, batch_size=1024, epochs=1):

    metadata = pickle.load(open(Path(metadata_dir) / 'metadata_stats.pkl', 'rb'))
    print("Data stats:")
    print(f"Number of users: {metadata['n_users']}")
    print(f"Number of posts: {metadata['n_posts']}")
    print(f"User categorical dims: {metadata['user_cat_dims']}")
    print(f"Post categorical dims: {metadata['post_cat_dims']}")
    print(f"User numerical dim: {metadata['user_num_dim']}")
    print(f"Post numerical dim: {metadata['post_num_dim']}")

    model_struct = build_model_from_dims(
        n_users=metadata['n_users'], 
        n_posts=metadata['n_posts'],
        user_cat_dims=metadata['user_cat_dims'],
        post_cat_dims=metadata['post_cat_dims'],
        user_num_dim=metadata['user_num_dim'],
        post_num_dim=metadata['post_num_dim'],
        **model_args
    )
    print("Model structure:")
    print(model_struct)
    
    trainer = TwoTowerTrainer(model_struct, dataset=None, **trainer_args)

    print("Run model training:")
    model = run_parquet_streaming(
        parquet_dir=parquet_dir,
        model=model_struct,
        trainer=trainer,
        batch_size=batch_size,
        epochs=epochs,
        **data_args
    )
    return model

Run training pipeline

In [0]:
# Get pipeline parameters
model_config_path = dbutils.widgets.get("model_config_path")
with open(model_config_path, "r") as file:
    model_config = json.load(file)

In [0]:
data_args = dict(
    user_id_col='user_id',
    post_id_col='post_id',
    user_cat_cols=["ip_province", "vehicle_series", "platform"],
    user_num_cols=["months_since_registration", "months_since_consent", "identity", "is_koc", "has_profile_photo",
    "community_visits", "posts_published", "posts_viewed", "posts_liked", 
    "posts_commented", "posts_shared", "users_followed", "tribes_joined"],
    post_cat_cols=["post_type"],
    post_num_cols=["days_since_published", "is_ugc", "is_sink", "has_pic", "has_video",
            "views", "likes", "comments", "shares", "dwell_time"],
    user_emb_col='lastn_embs',
    post_emb_col='post_content_emb',
    label_col='label'
)

parquet_dir = model_config['TWO_TOWER_TRAIN_PARQUET_PATH']
metadata_dir = model_config['TWO_TOWER_METADATA_PATH']

model_args = model_config['model_args']
trainer_args = model_config['trainer_args']
batch_size = model_config['batch_size']
epochs = model_config['epochs']

In [0]:
model = run_model_training(metadata_dir, parquet_dir, model_args, trainer_args, data_args, batch_size=batch_size, epochs=epochs)

In [0]:
# save model
model_path = model_config['TWO_TOWER_MODEL_PATH']
torch.save(model.state_dict(), model_path)